# FST-Arch: run all experiments

This notebook reproduces the simulation section of the article **Growing and pruning neural networks by fixed-sequence testing with family-wise error control**.

Steps:
1. Get the code.
2. Compute the Monte Carlo *true* risks $\mu(a)$ of every architecture (`oracle.py`).
3. Choose the margins $\gamma$ (growing) and $\delta$ (pruning) and check which null hypotheses are true.
4. Run the replications (`sim.py`) and print Tables 1 and 2.
5. Plot the results.

Only `numpy`, `scipy` and `matplotlib` are needed; all are preinstalled on Colab. No GPU is needed.

## 1. Get the code

In [ ]:
import os, sys
if not os.path.exists('fst-architecture-selection'):
    !git clone -q https://github.com/yamenetoo/fst-architecture-selection.git
%cd fst-architecture-selection/experiments
!pip -q install -r ../requirements.txt
sys.path.insert(0, os.getcwd())

## 2. True architecture risks

For one fixed training sample, $\mu(a)$ is estimated with 300 seeds and 40 000 test points for every width $\{4,\dots,256\}$ and depth $\{1,2,3\}$, and for every pruning level of the width-256 network. This is the slowest step (a few minutes on Colab).

In [ ]:
import time
t0 = time.time()
!python oracle.py
print(f'oracle time: {time.time() - t0:.0f} s')

## 3. Choose the margins

$\gamma$ is the smallest improvement worth growing for and $\delta$ the largest loss accepted when pruning, both in units of the loss (MSE). Pick them so that the width chain has **both** true and false null hypotheses; otherwise the FWER column is trivially 0. The cell below prints the truth table for your choice.

In [ ]:
B = 1000        # Monte Carlo replications (use 100 for a quick test)
GAMMA = 0.005   # growing margin
DELTA = 0.01    # pruning margin

import numpy as np
from oracle import WIDTHS, DEPTHS, FRACS
orc = np.load('oracle.npz')
mu, mu_pr = orc['risk'].mean(0), orc['prune'].mean(0)

print('Growing, width chain at depth 1: H0 true if mu(w+) - mu(w) >= -gamma')
for i in range(len(WIDTHS) - 1):
    th = mu[i + 1, 0] - mu[i, 0]
    print(f'  {WIDTHS[i]:>3} -> {WIDTHS[i+1]:>3}: theta = {th:+.4f}   H0 {"TRUE" if th >= -GAMMA else "false"}')
print('\nPruning from width 256: H0 true if mu(a_k) - mu(a_0) >= delta')
for k, fr in enumerate(FRACS[1:], 1):
    h = mu_pr[k] - mu_pr[0]
    print(f'  remove {fr:.0%}: loss = {h:+.4f}   H0 {"TRUE" if h >= DELTA else "false"}')

## 4. Run the replications

Each replication draws new validation splits and seeds and runs the growing procedure with five standard errors and the pruning procedure with two margins.

In [ ]:
t0 = time.time()
!python sim.py {B} {GAMMA} {DELTA}
print(f'simulation time: {time.time() - t0:.0f} s')

In [ ]:
print('Table 1 rows (growing) -- paste into the article:\n')
print(open('table_building.tex').read())
print('Table 2 rows (pruning, scenarios A and B):\n')
print(open('table_pruning.tex').read())

## 4b. Exact size across validation size $n$ and number of seeds $R$

Each width pair is tested at the exact boundary $c=\theta_{\text{true}}$, so a calibrated test rejects in 5% of replications. Proposition 1 predicts that the seed-only error over-rejects when $n$ is small or $R$ is large, and the single-seed error over-rejects at every $n$.

In [ ]:
B_GRID = 500   # use 100 for a quick test
t0 = time.time()
!python grid.py {B_GRID}
print(f'grid time: {time.time() - t0:.0f} s')
print(open('table_grid.tex').read())

## 5. Plots

In [ ]:
import json
import matplotlib.pyplot as plt
res = json.load(open('results.json'))
grid = json.load(open('grid.json'))

names = ['crossed', 'crossed+safety', 'seed_only', 'single', 'welch']
labels = ['Crossed\n(proposed)', 'Crossed\n+ safety', 'Seed-only', 'Single\nseed', 'Welch']
fwer = [res['build'][m]['fwer'] for m in names]
se = [res['build'][m]['fwer_se'] for m in names]

fig, ax = plt.subplots(1, 3, figsize=(16, 4.2))
ax[0].bar(labels, fwer, yerr=[1.96 * s for s in se], color='#4c72b0')
ax[0].axhline(0.05, ls='--', color='k', lw=1, label='nominal 0.05')
ax[0].set_title('Growing: family-wise error rate'); ax[0].legend()

methods = ['crossed', 'seed_only', 'single', 'welch']
for m, c in zip(methods, ['#4c72b0', '#dd8452', '#c44e52', '#8172b3']):
    for R, ls in [(5, ':'), (10, '-'), (20, '--')]:
        ys = [grid[f'n={n},R={R}'][m] for n in [50, 100, 250, 1000]]
        ax[1].plot([50, 100, 250, 1000], ys, ls=ls, color=c, marker='o', label=f'{m}, R={R}' if R == 10 else None)
ax[1].axhline(0.05, color='k', lw=1); ax[1].set_xscale('log')
ax[1].set_xlabel('validation size n'); ax[1].set_title('Exact size at the boundary (: R=5, - R=10, -- R=20)')
ax[1].legend(fontsize=8)

for key, off, lab in [('prune', -0.2, 'A: width 256'), ('prune2', 0.2, 'B: width 64')]:
    pe = [res[key][k]['harm_exceeds'] for k in ['cumulative', 'stepwise']]
    ps = [res[key][k]['harm_exceeds_se'] for k in ['cumulative', 'stepwise']]
    ax[2].bar(np.arange(2) + off, pe, width=0.4, yerr=[1.96 * s for s in ps], label=lab)
ax[2].set_xticks([0, 1]); ax[2].set_xticklabels(['Cumulative\n(proposed)', 'Stepwise'])
ax[2].axhline(0.05, ls='--', color='k', lw=1); ax[2].legend()
ax[2].set_title(r'Pruning: $P\{\mu(\hat a)-\mu(a_0)\geq\delta\}$')
plt.tight_layout(); plt.savefig('results.png', dpi=200); plt.show()

print('Exact size at the boundary in the main design (n=250, R=10), mean / max over width pairs:')
for m, v in res['boundary'].items():
    print(f"  {m:<10} {v['mean']:.3f} / {v['max']:.3f}")

## 6. Download the results

`results.json`, `grid.json`, the three `table_*.tex` files and `results.png` are in the `experiments` folder (Files panel on the left).

In [ ]:
try:
    from google.colab import files
    for f in ['results.json', 'grid.json', 'table_building.tex', 'table_pruning.tex', 'table_grid.tex', 'results.png']:
        files.download(f)
except ImportError:
    print('Not on Colab: files are in', os.getcwd())